# 표 파운데이션 모델과 소재 데이터 실습

합성 데이터 실습입니다. 실제 소재 물성을 나타내지 않습니다. 기본 모델은 실행 검증했으며 TabPFN 추론은 미검증입니다. 전체 해설은 함께 제공한 Markdown을 참고하세요.

## 1 설치
아래 셀은 필요할 때만 실행합니다. TabPFN은 별도로 설치하며 최초 사용 시 체크포인트 다운로드가 필요할 수 있습니다.

In [ ]:
# %pip install numpy pandas scikit-learn
# %pip install tabpfn

## 2 데이터와 평가 함수
이 셀은 독립 실행에 필요한 전체 함수를 정의합니다. 파일 다운로드나 학습은 아직 하지 않습니다.

In [ ]:
"""교육용 합성 소재 데이터 비교. 기본 실행은 TabPFN 다운로드 없이 동작합니다."""
import argparse
import json
import platform
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def make_materials(n=240, seed=42):
    """실제 NCM 물성식이 아닙니다. 단위와 변수만 소재 실험을 본뜬 장난감입니다."""
    rng = np.random.default_rng(seed)
    ni = rng.uniform(0.50, 0.95, n)
    temp = rng.uniform(680, 850, n)
    hours = rng.uniform(6, 20, n)
    X = pd.DataFrame({"ni_fraction": ni, "temperature_C": temp, "hours": hours})
    y = (160 + 65 * (ni - 0.5) - 0.002 * (temp - 770)**2
         + 5 * np.log(hours / 6) + 0.08 * (ni - 0.7) * (temp - 770)
         + rng.normal(0, 3, n))
    return X, pd.Series(y, name="capacity_mAh_g")


def make_models():
    kernel = ConstantKernel(1.0, (0.01, 100)) * Matern(
        length_scale=np.ones(3), length_scale_bounds=(0.01, 100), nu=2.5
    ) + WhiteKernel(0.1, (1e-5, 10))
    return {
        "Mean": DummyRegressor(),
        "RandomForest": RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                                               random_state=42, n_jobs=1),
        "HistGBDT": HistGradientBoostingRegressor(max_iter=150, max_leaf_nodes=15,
                                                  min_samples_leaf=10, random_state=42),
        "GPR": make_pipeline(StandardScaler(), GaussianProcessRegressor(
            kernel=kernel, normalize_y=True, random_state=42)),
    }


def run_benchmark(include_tabpfn=False, device="cpu", seeds=(42, 43, 44)):
    X, y = make_materials()
    idx = np.arange(len(X))
    splits = [(f"random_{s}", *train_test_split(idx, test_size=0.25, random_state=s))
              for s in seeds]
    # 평가 전에 임계값을 고정하고 고Ni 영역을 학습에서 제외합니다。
    splits.append(("high_Ni_holdout", idx[X.ni_fraction < 0.84], idx[X.ni_fraction >= 0.84]))
    rows = []
    for split, train, test in splits:
        models = {name: clone(model) for name, model in make_models().items()}
        if include_tabpfn:
            from tabpfn import TabPFNRegressor
            from tabpfn.constants import ModelVersion
            # 논문 버전과 현재 기본값의 혼동을 줄이기 위해 v2를 지정합니다.
            models["TabPFN_v2"] = TabPFNRegressor.create_default_for_version(
                ModelVersion.V2, device=device)
        for name, model in models.items():
            start = perf_counter()
            model.fit(X.iloc[train], y.iloc[train])
            fit_s = perf_counter() - start
            start = perf_counter()
            pred = model.predict(X.iloc[test])
            predict_s = perf_counter() - start
            rows.append(dict(split=split, model=name, n_train=len(train), n_test=len(test),
                             MAE=mean_absolute_error(y.iloc[test], pred),
                             RMSE=float(np.sqrt(mean_squared_error(y.iloc[test], pred))),
                             R2=r2_score(y.iloc[test], pred), fit_s=fit_s, predict_s=predict_s))
    return pd.DataFrame(rows), X, y


def environment():
    result = {"python": platform.python_version(), "platform": platform.platform()}
    for name in ["numpy", "pandas", "scikit-learn", "torch", "tabpfn"]:
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = "not installed"
    return result




## 3 기본 모델 비교
동일한 분할을 모든 모델에 적용합니다. GPR 최적화 경고가 나오면 결과 해석에 기록하세요.

In [ ]:
scores, X, y = run_benchmark()
display(X.assign(capacity_mAh_g=y).head())
display(scores.round(3))

## 4 분할별 평균 오차
무작위 분할 세 번은 같은 데이터의 재분할입니다. 독립 데이터셋 세 개의 결과가 아닙니다.

In [ ]:
summary = scores.assign(scenario=scores["split"].str.replace(r"random_\d+", "random", regex=True))
display(summary.groupby(["scenario", "model"])[["MAE", "RMSE", "R2"]].mean().round(3))

## 5 TabPFN 추가
아래 스위치를 True로 바꾸면 TabPFN v2를 포함해 전체 비교를 실행합니다. CPU 또는 준비된 CUDA 환경을 선택하세요. 이 단계는 작성 환경에서 실행하지 않았습니다.

In [ ]:
RUN_TABPFN = False
if RUN_TABPFN:
    scores_tabpfn, _, _ = run_benchmark(include_tabpfn=True, device="cpu")
    display(scores_tabpfn.round(3))
else:
    print("TabPFN 비교를 건너뜁니다. 설치 후 RUN_TABPFN=True로 변경하세요.")

## 6 실행 환경 기록
결과와 함께 사용 패키지 및 모델 버전을 남기세요.

In [ ]:
print(json.dumps(environment(), indent=2))

## 생각해 볼 점

1. 고Ni 영역을 제외하면 오차가 왜 커질까요?
2. 매끄러운 합성 함수에서 GPR이 유리한 결과를 실제 데이터로 일반화할 수 있을까요?
3. 반복 측정이 학습과 평가에 섞이면 어떤 문제가 생길까요?

[TabPFN 공식 코드](https://github.com/PriorLabs/TabPFN) · [원리 논문](https://arxiv.org/abs/2207.01848) · [TabICL](https://arxiv.org/abs/2502.05564)